# Pipeline Control Notebook
Use these interactive cells to run the Scraping, Transformation, and Ingestion stages manually.


In [ ]:
import os
import sys
from pathlib import Path

# Fix python path to allow imports from src
current_dir = Path().absolute()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.forensics.manual_scrapping import scrape_submission_url
from src.infra.data_transformation import transform_to_structured
from src.infra.gcp_ingestion import load_to_bigquery


## 1. Scrape a Reddit Thread
Replace the URL below with the submission you want to scrape.

In [ ]:
TARGET_URL = "https://www.reddit.com/r/mexico/comments/1rcb63u/i_hope_you_win_your_fight_against_the_cartel/"

print(f"Starting manual scrape for: {TARGET_URL}")
raw_file_path = scrape_submission_url(TARGET_URL)

if raw_file_path:
    print(f"Scraping complete. Raw file saved at: {raw_file_path}")
else:
    print("Scraping failed.")

## 2. Transform Phase
Define your mode:
- `master`: Transforms all JSONs into a single master CSV (replaces BigQuery table entirely).
- `single`: Transforms only the latest raw file (appends to BigQuery table).

In [ ]:
MODE = "master"
print(f"Starting transformation in {MODE} mode...")

if MODE == "master":
    print("Consolidating all raw data into master CSV...")
    structured_file = transform_to_structured(
        input_path="data/raw", 
        output_dir="data/structured", 
        format="csv"
    )
    write_disposition = "WRITE_TRUNCATE"
else:
    print(f"Transforming single submission: {raw_file_path}")
    structured_file = transform_to_structured(
        input_path=raw_file_path, 
        output_dir="data/structured", 
        format="csv"
    )
    write_disposition = "WRITE_APPEND"

if structured_file:
    print(f"Transformation complete. Structured file ready at: {structured_file}")
else:
    print("Transformation failed.")

## 3. Ingestion Phase
Push the structured data to BigQuery.

In [ ]:
print(f"Syncing {structured_file} to BigQuery...")
print(f"Table operation mode: {write_disposition}")

try:
    load_to_bigquery(
        file_path=structured_file, 
        table_name="comments_structured", 
        write_disposition=write_disposition
    )
    print("Ingestion complete. Data is now in GCP.")
except Exception as e:
    print(f"Ingestion failed: {e}")